In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import warnings
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"XGBoost version: {xgb.__version__}")

Libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 2.0.2
XGBoost version: 3.2.0


In [4]:
# Source: Arsiwala et al. 2025
# Features to use: HIC RT, SMAC RT, HAC RT, PR Ova, PR CHO, SEC %Monomer, AC-SINS pH 6.0, AC-SINS pH 7.4, Tm1, Tm2, Titer
# Prediction to make: Approved (1) vs Not-Approved (0) — 106 approved, 91 not-approved
# Boostrap 20 independent XGBoost models, random 80:20 train-test splits

df = pd.read_csv('GDPa1_v1.2_20250814.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nClinical status distribution:')
print(df['highest_clinical_trial_asof_feb2025'].value_counts())

Dataset shape: (246, 30)

Clinical status distribution:
highest_clinical_trial_asof_feb2025
Approved                106
Phase-II                 80
Phase-III                39
Phase-I                  13
Preregistration (w)       2
Phase-II/III              2
Preregistration           2
Phase-I/II                1
Approved (withdrawn)      1
Name: count, dtype: int64


In [5]:
FEATURE_COLS = [
    'HIC',           # Hydrophobic interaction chromatography RT
    'SMAC',          # Standup monolayer affinity chromatography RT
    'HAC',           # Heparin affinity chromatography RT
    'PR_Ova',        # Polyreactivity score (ovalbumin)
    'PR_CHO',        # Polyreactivity score (CHO membrane proteins)
    'SEC %Monomer',  # Size-exclusion chromatography % monomer
    'AC-SINS_pH6.0', # Self-association (AC-SINS) at pH 6.0
    'AC-SINS_pH7.4', # Self-association (AC-SINS) at pH 7.4
    'Tm1',           # First melting temperature (nanoDSF)
    'Tm2',           # Second melting temperature (nanoDSF)
    'Titer',         # Expression titer (µg/mL)
]

approved_mask = df['highest_clinical_trial_asof_feb2025'] == 'Approved'
not_approved_mask = ((df['est_status_asof_feb2025'] == 'Discontinued') & ~approved_mask)

df_model = df[approved_mask | not_approved_mask].copy()
df_model['label'] = approved_mask[approved_mask | not_approved_mask].astype(int)

print(f'Antibodies for modeling: {len(df_model)}')
print(f'Approved: {df_model["label"].sum()}')
print(f'Not-approved: {(df_model["label"] == 0).sum()}')

Antibodies for modeling: 198
  Approved:     106
  Not-approved: 92


In [7]:
X_full = df_model[FEATURE_COLS].copy()
missing_before = X_full.isna().sum().sum()

# Use median for imputation
for col in FEATURE_COLS:
    X_full[col] = X_full[col].fillna(X_full[col].median())

print(f'Missing values imputed: {missing_before}')

y_full = df_model['label'].values
antibody_names_full = df_model['antibody_name'].values

print(f'Feature matrix: {X_full.shape[0]} antibodies, {X_full.shape[1]} features')
print(f'Feature summary: ')
X_full.describe().round(2)

Missing values imputed: 280
Feature matrix: 198 antibodies, 11 features
Feature summary: 


,HIC,SMAC,HAC,PR_Ova,PR_CHO,SEC %Monomer,AC-SINS_pH6.0,AC-SINS_pH7.4,Tm1,Tm2,Titer
count,198.00,198.00,198.00,198.00,198.00,198.00,198.00,198.00,198.00,198.00,198.00
mean,2.83,2.99,3.67,0.11,0.16,95.68,1.85,5.57,70.64,82.45,236.39
std,0.33,0.89,0.72,0.12,0.14,4.10,4.27,7.97,1.96,2.59,110.28
min,2.43,2.68,1.00,0.00,0.00,75.30,-0.40,-1.88,64.05,75.04,34.26
25%,2.58,2.72,3.78,0.02,0.04,94.65,0.35,0.00,69.64,81.40,154.18
50%,2.76,2.75,3.78,0.10,0.14,97.38,0.75,1.50,70.51,82.88,227.63
75%,2.99,2.87,3.78,0.11,0.27,98.24,1.36,10.09,71.64,84.14,301.96
max,4.50,10.00,5.48,0.60,0.55,99.86,30.60,29.50,76.24,88.57,642.34


In [ ]:
def build_xgboost_model(random_state=None):
    """
    XGBoost model matching paper hyperparameters.

    Paper XGBoost params:
        max_depth=25
        learning_rate=0.25
        n_estimators=200
        subsample=0.8
        reg_alpha=0.1
        reg_lambda=0.1
    """
    return XGBClassifier(
        max_depth=25,
        learning_rate=0.25,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist"
    )